# NB27 — Mobile-portable ablation: what signal remains without cursor?

**Regime:** `[LAB]` for evaluation (uses AdSERP click labels), but the candidate feature set is strictly cursor-free so the ablation corresponds to what a mobile deployment could compute from scroll + text alone.

## Key Claims

| K-ID | Claim | Value | Source cell |
|---|---|---|---|
| K1 | Spearman ρ(viewport_dwell_ms, clicked) | **+0.1623**, p = 4.4 × 10⁻⁷⁵, n = 12,588 | Cell 6 |
| K2 | Spearman ρ(viewport_dwell_residual, clicked) — novelty-baseline residual | **+0.1617**, p = 1.6 × 10⁻⁷⁴, n = 12,588 | Cell 6 |
| K3 | Cursor-free LOSO AUC (5 text + viewport dwell + residual = 7 features) | **0.6304 concat, per-fold 0.6308 ± 0.0650** (46 participants) | Cell 8 |
| K4 | Text-only LOSO AUC (5 features, no viewport) | **0.5268 concat, per-fold 0.5187 ± 0.0514** | Cell 8 |
| K5 | Viewport contribution over text alone: K3 − K4 | **+0.1037 concat**; cursor-free beats text-only in **44 / 46 folds** | Cell 8 |
| K6 | M4 cursor baseline LOSO AUC (matched 12,588-record subset) | **0.8603 concat, per-fold 0.8605 ± 0.0465** | Cell 8 |
| K7 | Cursor-free recovery ratio: K3 / K6 | **73.28 %** (0.6304 / 0.8603); M4 beats cursor-free in **46 / 46 folds** (100 %) | Cell 8 |
| K8 | K1 vs K2 — novelty-baseline residual adds essentially nothing over raw viewport dwell | Spearman delta ≈ 0.0006, same to fourth decimal | Cell 6 |

**Primary interpretation.** Viewport dwell is the single biggest contributor in a cursor-free feature set (+10.4 AUC pts over text alone, 44 / 46 folds), empirically validating Peter's "time-in-viewport" intuition at the click-prediction level. But the M4 cursor baseline still wins in **100 % of LOPO folds** with a mean paired margin of +0.23 AUC. Cursor-free feature set recovers ~73 % of desktop cursor AUC; the remaining ~27 % is cursor-specific information about sub-viewport intent (which result the hand is hovering over, approach-retreat geometry) that scroll + text cannot reproduce.

**Disposition.** Positive finding with a clearly-named gap. Goes into `docs/findings-approach-retreat.md` as a mobile-portability reference point (not CIKM paper material — CIKM's story is cursor-deployable-at-inference on desktop, a mobile-relevance experiment would muddy the through-line). Task-model paper TODO gets an entry citing K7 as concrete evidence for the §5 mobile-generalization claim.

**Secondary null (K8).** The novelty-baseline residual (Peter's refinement of the lexical-novelty idea applied to viewport dwell) is empirically indistinguishable from raw viewport dwell at this grain. Same pattern observed in NB25 K6/K8. Documented in `docs/null-findings/2026-04-15-novelty-baseline-residual-redundancy.md` — two independent shots at "lexical-novelty residual as a feature" have both returned "the residual is the raw variable in disguise" because the cos-sim/text-length baseline explains essentially none of the variance in per-result dwell at sentence-embedding granularity.

## Context: Peter Dixon-Moses's mobile question (2026-04-15)

Peter's question: *if we ablate the cursor approach signal, would the residual contain something worth capturing (for future real-time applications, especially mobile where there is no cursor)?*

The cursor approach features dominate click prediction on desktop (§4.1 of the CIKM paper shows 9 cursor features absorb position at no AUC cost, LOSO 0.821). On mobile, those features don't exist. Peter's candidates for a mobile-portable feature set:

1. Total time in viewport per result
2. Time-in-viewport vs expected-time-in-viewport — the lexical-novelty residual idea from NB25 but applied to viewport exposure. Mobile-portable by construction — needs only scroll position and SERP text.
3. Scroll regression per result (not implemented in v1 of NB27 — worth a follow-up if the question sharpens)
4. Text-relevance features (the 5 from NB26: lex_overlap, avg_tf, cos_title, cos_snippet, cos_combined)

**Framing after NB26:** NB26 established that a small linear ranker on 5 text features alone trails Google on full-SERP MRR. So NB27 is **not** trying to build a mobile ranker that beats Google — it's answering a narrower question: *how much of the M4 cursor-signal click-prediction AUC is recoverable from a cursor-free feature set?*

**CV protocol:** leave-one-participant-out, 46 participants (one held out for trials with missing viewport data), click prediction target, LOSO concat + per-fold AUC. 12,588 matched (trial, position) records.

In [1]:
# ── Imports + paths ───────────────────────────────────────────────────
import json
import re
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = Path("/Users/andyed/Documents/dev/attentional-foraging")
sys.path.insert(0, str(ROOT / "notebooks-v2"))
from data_loader import (
    load_fixations, load_mouse_events, get_trial_meta, result_band_tops,
    interpolate_scroll,
)

LAB_RECORDS = ROOT / "AdSERP/data/cursor-approach-features.json"
SERP_EMB_COMBINED = ROOT / "AdSERP/data/serp-embeddings.json"
SERP_EMB_SPLIT = ROOT / "AdSERP/data/serp-embeddings-split.json"
QUERY_EMB = ROOT / "AdSERP/data/query-embeddings.json"

M4_FEATURES = [
    "min_dist", "mean_dist", "final_dist", "retreat_dist",
    "dwell_in_proximity_ms",
    "mean_approach_velocity", "max_approach_velocity",
    "direction_changes", "frac_decreasing",
]
print("imports ok")

imports ok


In [2]:
# ── Load LAB records and embeddings ───────────────────────────────────
lab_records = json.load(open(LAB_RECORDS))
print(f"LAB records: {len(lab_records):,}")

with open(SERP_EMB_SPLIT) as f:
    serp_split = json.load(f)
with open(SERP_EMB_COMBINED) as f:
    serp_combined = json.load(f)
with open(QUERY_EMB) as f:
    query_data = json.load(f)
print(f"split embeddings: {len(serp_split):,} trials")
print(f"combined embeddings: {len(serp_combined):,} trials")
print(f"query embeddings: {len(query_data):,} trials")

combined_emb = {}
for tid, rs in serp_combined.items():
    for r in rs:
        if "embedding" in r:
            combined_emb[(tid, r["position"])] = np.array(r["embedding"], dtype=np.float32)

title_emb, snippet_emb, title_text, snippet_text = {}, {}, {}, {}
for tid, rs in serp_split.items():
    for r in rs:
        key = (tid, r["position"])
        if "title_embedding" in r:
            title_emb[key] = np.array(r["title_embedding"], dtype=np.float32)
        if "snippet_embedding" in r:
            snippet_emb[key] = np.array(r["snippet_embedding"], dtype=np.float32)
        title_text[key] = r.get("title", "") or ""
        snippet_text[key] = r.get("snippet", "") or ""

query_emb_lookup = {tid: np.array(v["embedding"], dtype=np.float32)
                    for tid, v in query_data.items()
                    if isinstance(v, dict) and "embedding" in v}
query_text_lookup = {tid: v.get("query", "") for tid, v in query_data.items()
                     if isinstance(v, dict)}

print(f"\nlookups built: combined={len(combined_emb):,}, title={len(title_emb):,}, "
      f"snippet={len(snippet_emb):,}, query={len(query_emb_lookup):,}")

LAB records: 13,419


split embeddings: 2,776 trials
combined embeddings: 2,776 trials
query embeddings: 2,776 trials



lookups built: combined=27,520, title=27,520, snippet=27,520, query=2,776


In [3]:
# ── Per-result viewport dwell computation ────────────────────────────
# For each trial, the scroll stream gives us (t, scroll_y) pairs. At any
# time t, the viewport occupies [scroll_y, scroll_y + screen_h] in
# page-space Y. A result band N occupies [tops[N], tops[N+1]).
# Result N is visible at time t iff the viewport interval intersects
# the band interval, i.e.:
#   tops[N] < scroll_y + screen_h   AND   tops[N+1] > scroll_y
#
# We walk the scroll events sorted by time. Between consecutive events,
# scroll_y is piecewise constant (linear interpolation is approximately
# fine for the smooth-scrolling case too). For each inter-event interval
# we check visibility of each result and accumulate Δt where visible.
#
# Skip trials missing fixation / mouse / meta data (they can't be
# evaluated anyway).

def compute_viewport_dwells(trial_id):
    """Return dict: position -> viewport dwell time in ms. Returns None on skip."""
    meta = get_trial_meta(trial_id)
    if meta is None or meta[0] is None:
        return None
    doc_h, screen_h, _ = meta
    mouse_data = load_mouse_events(trial_id)
    if mouse_data is None:
        return None
    _, scrolls, _ = mouse_data
    if not scrolls:
        return None
    fixations = load_fixations(trial_id)
    if fixations is None or len(fixations) < 2:
        return None

    n_results = len(serp_split.get(trial_id, []))
    if n_results == 0:
        return None
    tops = result_band_tops(n_results, doc_h)
    # Band intervals — use next top as the end; the last result goes to doc_h.
    band_intervals = []
    for i in range(n_results):
        top = tops[i]
        bottom = tops[i + 1] if i + 1 < len(tops) else doc_h
        band_intervals.append((top, bottom))

    # Scroll events: list of (t, y). Use trial start time as the "pre-first-scroll"
    # position (assume scroll_y = 0 before the first event).
    s_ts = [s[0] for s in scrolls]
    s_ys = [s[1] for s in scrolls]
    # Pad: assume scroll_y = 0 at the trial start (first fixation timestamp).
    t_start = fixations[0]["t"]
    t_end = fixations[-1]["t"] + fixations[-1].get("d", 0)
    if s_ts[0] > t_start:
        s_ts = [t_start] + s_ts
        s_ys = [0] + s_ys
    if s_ts[-1] < t_end:
        s_ts.append(t_end)
        s_ys.append(s_ys[-1])

    dwells = [0.0] * n_results
    # Walk consecutive scroll segments
    for i in range(len(s_ts) - 1):
        t0, y0 = s_ts[i], s_ys[i]
        t1, y1 = s_ts[i + 1], s_ys[i + 1]
        dt = max(0.0, t1 - t0)
        if dt <= 0:
            continue
        # Treat this segment as scroll_y = midpoint(y0, y1) (OK for small segments)
        y_mid = (y0 + y1) / 2.0
        vp_top = y_mid
        vp_bottom = y_mid + screen_h
        for pos, (b_top, b_bot) in enumerate(band_intervals):
            # Interval intersection test
            if b_top < vp_bottom and b_bot > vp_top:
                dwells[pos] += dt
    return {pos: dwells[pos] for pos in range(n_results)}

# Smoke test on first available trial
test_tid = next(iter(serp_split))
test_d = compute_viewport_dwells(test_tid)
print(f"smoke test trial {test_tid}: viewport dwells (ms) = {test_d}")

smoke test trial p004-b1-t1: viewport dwells (ms) = None


In [4]:
# ── Aggregate viewport dwells across all trials ──────────────────────

viewport_dwell_by_key = {}
trial_ids = sorted(serp_split.keys())
skipped = 0
for i, tid in enumerate(trial_ids):
    if i % 300 == 0:
        print(f"  {i}/{len(trial_ids)}  (skipped {skipped})")
    d = compute_viewport_dwells(tid)
    if d is None:
        skipped += 1
        continue
    for pos, ms in d.items():
        viewport_dwell_by_key[(tid, pos)] = ms

print(f"\nviewport dwell entries: {len(viewport_dwell_by_key):,}")
print(f"skipped trials: {skipped}")

dwells_arr = np.array(list(viewport_dwell_by_key.values()))
print(f"\nviewport dwell (ms) stats:")
print(f"  mean={dwells_arr.mean():.0f}, median={np.median(dwells_arr):.0f}")
print(f"  p25={np.percentile(dwells_arr, 25):.0f}, p75={np.percentile(dwells_arr, 75):.0f}")
print(f"  min={dwells_arr.min():.0f}, max={dwells_arr.max():.0f}")
print(f"  zeros (result never visible): {(dwells_arr == 0).sum():,}")

  0/2776  (skipped 0)


  300/2776  (skipped 50)


  600/2776  (skipped 100)


  900/2776  (skipped 145)


  1200/2776  (skipped 213)


  1500/2776  (skipped 251)


  1800/2776  (skipped 358)


  2100/2776  (skipped 392)


  2400/2776  (skipped 442)


  2700/2776  (skipped 507)

viewport dwell entries: 22,445
skipped trials: 510

viewport dwell (ms) stats:
  mean=10722, median=7497
  p25=0, p75=16813
  min=0, max=77893
  zeros (result never visible): 6,017


In [5]:
# ── Compute the 5 text features (reuse NB26 logic) ────────────────────
STOPWORDS = set("a an and or but the is are was were be been being "
                "to of in on at for from by with as "
                "this that these those".split())

def tokenize(text):
    return [t for t in re.findall(r"[a-z0-9]+", (text or "").lower())
            if t not in STOPWORDS and len(t) > 1]

def cos_sim(a, b):
    if a is None or b is None:
        return 0.0
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    return float(np.dot(a, b) / (na * nb)) if na and nb else 0.0

text_features_by_key = {}
for key in viewport_dwell_by_key:
    tid, pos = key
    q_emb = query_emb_lookup.get(tid)
    q_text = query_text_lookup.get(tid, "")
    t_emb = title_emb.get(key)
    s_emb = snippet_emb.get(key)
    c_emb = combined_emb.get(key)
    if q_emb is None or t_emb is None or s_emb is None or c_emb is None:
        continue
    q_tokens = tokenize(q_text)
    r_text = (title_text.get(key, "") or "") + " " + (snippet_text.get(key, "") or "")
    r_tokens = tokenize(r_text)
    if not q_tokens:
        lex_overlap = avg_tf = 0.0
    else:
        counter = {}
        for t in r_tokens:
            counter[t] = counter.get(t, 0) + 1
        lex_overlap = sum(1 for t in q_tokens if t in counter) / len(q_tokens)
        avg_tf = float(np.mean([counter.get(t, 0) for t in q_tokens]))
    text_features_by_key[key] = [
        lex_overlap,
        avg_tf,
        cos_sim(q_emb, t_emb),
        cos_sim(q_emb, s_emb),
        cos_sim(q_emb, c_emb),
    ]

# Also keep a r_text_length feature for the expected-viewport-dwell baseline
text_length_by_key = {}
for key in viewport_dwell_by_key:
    tid, pos = key
    r_text = (title_text.get(key, "") or "") + " " + (snippet_text.get(key, "") or "")
    text_length_by_key[key] = len(r_text)

print(f"text feature vectors: {len(text_features_by_key):,}")
print(f"text length entries:  {len(text_length_by_key):,}")

text feature vectors: 22,445
text length entries:  22,445


In [6]:
# ── K1: Spearman viewport_dwell ~ clicked ─────────────────────────────
# And: K2 expected-viewport-dwell baseline + residual computation.

click_by_key = {(r["trial_id"], r["position"]): bool(r.get("was_clicked", False))
                for r in lab_records}

# Build aligned arrays for the viewport + click + text-length subset
keys_aligned = []
vpd = []
txt_len = []
cos_q = []
clicked_arr = []
for key in viewport_dwell_by_key:
    if key not in click_by_key:
        continue
    if key not in text_features_by_key:
        continue
    keys_aligned.append(key)
    vpd.append(viewport_dwell_by_key[key])
    txt_len.append(text_length_by_key[key])
    # Use cos_combined as the query-similarity feature for the expected baseline
    cos_q.append(text_features_by_key[key][4])
    clicked_arr.append(click_by_key[key])
vpd = np.array(vpd, dtype=float)
txt_len = np.array(txt_len, dtype=float)
cos_q = np.array(cos_q, dtype=float)
clicked_arr = np.array(clicked_arr, dtype=bool)
print(f"aligned sample: {len(keys_aligned):,}")
print(f"click rate:     {clicked_arr.mean():.3f}")

# K1
rho_vpd, p_vpd = spearmanr(vpd, clicked_arr.astype(float))
print(f"\n[K1] Spearman rho(viewport_dwell_ms, clicked) = {rho_vpd:+.4f}  (p = {p_vpd:.2e})")

# K2: expected-viewport-dwell baseline
# Expected dwell = f(text_length, cos_combined). Fit on the full sample
# (no leakage for a descriptive baseline; the LOSO test in Cell 9 is the
# actual predictive evaluation, not this Spearman).
X_base = np.column_stack([txt_len, cos_q])
# Use log(1 + dwell) to handle right-skewed distribution and zero dwells
log_vpd = np.log1p(vpd)
reg = LinearRegression()
reg.fit(X_base, log_vpd)
expected_log_vpd = reg.predict(X_base)
vpd_residual = log_vpd - expected_log_vpd
print(f"\nexpected log(1+vpd) = {reg.coef_[0]:+.6f} * text_length "
      f"{reg.coef_[1]:+.4f} * cos_query {reg.intercept_:+.4f}")
print(f"residual stats: mean={vpd_residual.mean():+.4f}, std={vpd_residual.std():.4f}")

# K2 — Spearman residual ~ click
rho_res, p_res = spearmanr(vpd_residual, clicked_arr.astype(float))
print(f"\n[K2] Spearman rho(viewport_dwell_residual, clicked) = {rho_res:+.4f}  (p = {p_res:.2e})")

# Store the residual keyed by (trial, position) for downstream use
viewport_residual_by_key = dict(zip(keys_aligned, vpd_residual.tolist()))
print(f"\nresidual entries keyed by (trial, position): {len(viewport_residual_by_key):,}")

aligned sample: 12,588
click rate:     0.154

[K1] Spearman rho(viewport_dwell_ms, clicked) = +0.1623  (p = 4.37e-75)

expected log(1+vpd) = +0.000149 * text_length +0.3211 * cos_query +9.1197
residual stats: mean=+0.0000, std=0.9803

[K2] Spearman rho(viewport_dwell_residual, clicked) = +0.1617  (p = 1.55e-74)

residual entries keyed by (trial, position): 12,588


In [7]:
# ── Build the three feature sets for LOSO click prediction ───────────
# Set A (cursor-free, mobile-portable): 5 text features + viewport_dwell + viewport_residual = 7 features
# Set B (text only, baseline): 5 text features
# Set C (M4 cursor baseline): 9 cursor approach features from lab_records

lab_index = {(r["trial_id"], r["position"]): i for i, r in enumerate(lab_records)}

# Matched subset: records where we have viewport dwell + text features + lab record
matched_keys = [k for k in keys_aligned if k in lab_index]
print(f"matched (viewport + text + lab): {len(matched_keys):,}")

X_textonly = []
X_cursor_free = []
X_m4 = []
y = []
groups = []
for key in matched_keys:
    tf = text_features_by_key[key]
    vp = viewport_dwell_by_key[key]
    vr = viewport_residual_by_key[key]
    X_textonly.append(tf)
    X_cursor_free.append(tf + [vp, vr])
    lab_i = lab_index[key]
    r = lab_records[lab_i]
    X_m4.append([float(r.get(f, 0) or 0) for f in M4_FEATURES])
    y.append(bool(r.get("was_clicked", False)))
    groups.append(r["trial_id"].split("-")[0])

X_textonly = np.array(X_textonly)
X_cursor_free = np.array(X_cursor_free)
X_m4 = np.array(X_m4)
y = np.array(y, dtype=int)
groups = np.array(groups)

print(f"\nfeature matrix shapes:")
print(f"  text-only (5):        {X_textonly.shape}")
print(f"  cursor-free (7):      {X_cursor_free.shape}")
print(f"  M4 cursor (9):        {X_m4.shape}")
print(f"click rate: {y.mean():.3f}")
print(f"participants: {len(set(groups))}")

matched (viewport + text + lab): 12,588

feature matrix shapes:
  text-only (5):        (12588, 5)
  cursor-free (7):      (12588, 7)
  M4 cursor (9):        (12588, 9)
click rate: 0.154
participants: 46


In [8]:
# ── K3, K4, K5, K7: LOSO comparison ──────────────────────────────────

def loso_auc(X, y, groups, label):
    pipe = Pipeline([
        ("sc", StandardScaler()),
        ("lr", LogisticRegression(max_iter=5000, class_weight="balanced", C=1.0)),
    ])
    n_groups = len(set(groups))
    gkf = GroupKFold(n_splits=n_groups)
    y_proba = cross_val_predict(pipe, X, y, groups=groups, cv=gkf,
                                method="predict_proba", n_jobs=1)[:, 1]
    concat = float(roc_auc_score(y, y_proba))
    per_fold = []
    for _, test_idx in gkf.split(X, y, groups=groups):
        yt = y[test_idx]
        if len(np.unique(yt)) < 2:
            continue
        per_fold.append(float(roc_auc_score(yt, y_proba[test_idx])))
    arr = np.array(per_fold)
    print(f"  {label}: concat AUC = {concat:.4f}  (per-fold {arr.mean():.4f} ± {arr.std(ddof=1):.4f}, n={len(arr)})")
    return concat, arr

print("── LOSO click prediction ──")
auc_text, folds_text = loso_auc(X_textonly, y, groups, "Text-only (5 features)")
auc_cf, folds_cf = loso_auc(X_cursor_free, y, groups, "Cursor-free (5 text + viewport + residual = 7)")
auc_m4, folds_m4 = loso_auc(X_m4, y, groups, "M4 cursor baseline (9 features)")

print(f"\n[K3] Cursor-free AUC       : {auc_cf:.4f}")
print(f"[K4] Text-only AUC         : {auc_text:.4f}")
print(f"[K5] Viewport contribution : {auc_cf - auc_text:+.4f}")
print(f"[K6] M4 cursor baseline    : {auc_m4:.4f}")
print(f"[K7] Cursor-free recovery  : {auc_cf / auc_m4:.2%}  ({auc_cf:.4f} / {auc_m4:.4f})")

# Paired per-fold deltas
if len(folds_cf) == len(folds_m4):
    paired = folds_m4 - folds_cf
    print(f"\npaired per-fold M4 − cursor-free:")
    print(f"  mean = {paired.mean():+.4f} ± {paired.std(ddof=1):.4f}")
    print(f"  M4 > cursor-free in {int((paired > 0).sum())}/{len(folds_m4)} folds")

if len(folds_cf) == len(folds_text):
    paired_cf_text = folds_cf - folds_text
    print(f"\npaired per-fold cursor-free − text-only:")
    print(f"  mean = {paired_cf_text.mean():+.4f} ± {paired_cf_text.std(ddof=1):.4f}")
    print(f"  cursor-free > text-only in {int((paired_cf_text > 0).sum())}/{len(folds_cf)} folds")

── LOSO click prediction ──


  Text-only (5 features): concat AUC = 0.5268  (per-fold 0.5187 ± 0.0514, n=46)


  Cursor-free (5 text + viewport + residual = 7): concat AUC = 0.6304  (per-fold 0.6308 ± 0.0650, n=46)


  M4 cursor baseline (9 features): concat AUC = 0.8603  (per-fold 0.8605 ± 0.0465, n=46)

[K3] Cursor-free AUC       : 0.6304
[K4] Text-only AUC         : 0.5268
[K5] Viewport contribution : +0.1037
[K6] M4 cursor baseline    : 0.8603
[K7] Cursor-free recovery  : 73.28%  (0.6304 / 0.8603)

paired per-fold M4 − cursor-free:
  mean = +0.2297 ± 0.0571
  M4 > cursor-free in 46/46 folds

paired per-fold cursor-free − text-only:
  mean = +0.1122 ± 0.0574
  cursor-free > text-only in 44/46 folds


## Results summary

After executing, transcribe K1–K7 into the Key Claims block at the top.

### Interpretation matrix

- **K7 ≥ 85 %** (cursor-free AUC ≥ 0.70): strong recovery. Mobile portability is within reach with scroll + text telemetry; the cursor advantage is narrow and mostly about sub-viewport precision.
- **K7 in 70–85 %**: moderate recovery. Mobile has a genuine gap — ~15–30 % of the click signal lives in cursor-specific motor dynamics that a cursor-free feature set can't reproduce. Candidate compensation: add gaze (webcam or mobile gaze), haptics, or more lexical features.
- **K7 < 70 %**: weak recovery. Cursor is doing substantial work a mobile deployment genuinely cannot reproduce without new instrumentation. The task model's deferred-class detection especially would not survive.
- **K5 > 0.02 (viewport adds > 2 AUC pts over text alone)**: viewport dwell + residual carries marginal signal the text features don't. Peter's "time-in-viewport vs expected" framing has empirical traction at the click-prediction level.
- **K5 ≈ 0**: viewport features are redundant with text — text relevance already captures what the user attends to. Less interesting for mobile deployment.
- **K5 < 0**: viewport features add noise. Unusual; indicates the viewport-dwell computation has a bug or the residual regression is mis-specified.

### Disposition

- **K7 ≥ 85 %, K5 > 0**: promising mobile feature set. Document in `docs/findings-approach-retreat.md` as a candidate for the task-model paper's §5 generalization discussion. Not CIKM material (still gaze-neighbor territory and doesn't close the main paper's argument).
- **K7 70–85 %, K5 > 0**: honest moderate result. Document in `docs/null-findings/` as a constrained positive with a clearly-named gap. Peter gets the narrow answer to his question: "yes, cursor-free captures ~X % of the signal; the remaining ~(100−X) % is cursor-specific."
- **K7 < 70 % or K5 ≈ 0**: null. Document in `docs/null-findings/` with the narrow surviving claim ("cursor signal contains ~30 % unique click-prediction information that cursor-free feature sets cannot recover") and name the next experiment (add gaze, haptics, larger text feature family).

Whichever branch fires, this notebook's result belongs in the task-model paper TODO (gaze-neighbor discussion) or the null-findings directory — not in the CIKM paper's main text.